In [1]:
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import os


In [2]:
import polars as pl
import time

#1. Modo Lazy: Estamos apenas escaneando o arquivo, sem carregar para a RAM
#scan_parquet já inicia em modo lazy por padrão.
q = (
    pl.scan_parquet("../aula12/202502_NovoBolsaFamilia_POLARS.parquet")
    # .rename({
    #     'M�S COMPET�NCIA': 'MES_COMPETENCIA',
    #     'M�S REFER�NCIA': 'MES_REFERENCIA',
    #     'C�DIGO MUNIC�PIO SIAFI': 'CODIGO_MUNICIPIO_SIAFI',
    #     'NOME MUNIC�PIO': 'NOME_MUNICIPIO',
    # })
    # .filter(pl.col("UF") == "SP")
)

print(type(q)) # Note que é um LazyFrame, não um DataFrame
print("\n")

# 2. O Plano de Execução (Explain)
print("--- Plano de Execução (Explain) ---")
print("O Polars analisou sua query e otimizou o caminho antes de tocar nos dados:")
print(q.explain())
print("\n")

# 3. Execução Real (Collect)
print("--- Executando (Collect) ---")
start_time = time.time()
df_resultado = q.collect()
end_time = time.time()

display(df_resultado)

<class 'polars.lazyframe.frame.LazyFrame'>


--- Plano de Execução (Explain) ---
O Polars analisou sua query e otimizou o caminho antes de tocar nos dados:


FileNotFoundError: O sistema não pode encontrar o arquivo especificado. (os error 2): ../aula12/202502_NovoBolsaFamilia_POLARS.parquet

In [ ]:
# Definindo a consulta estatística
stats_query = (
    pl.scan_parquet("../Aula12/202501_NovoBolsaFamilia_POLARS.parquet")
    .with_columns(
        pl.col("VALOR PARCELA")
        .str.replace(",", ".")     # "650,00" vira "650.00"
        .cast(pl.Float64)          # vira o número 650.0
    )
    .select([
        pl.col("VALOR PARCELA").min().alias("minimo"),
        pl.col("VALOR PARCELA").mean().alias("media"),
        pl.col("VALOR PARCELA").std().alias("desvio_padrao"),
        pl.col("VALOR PARCELA").max().alias("maximo"),
        pl.col("VALOR PARCELA").quantile(0.5).alias("mediana")
    ])
)

# Executando
print("--- Estatísticas Descritivas ---")
stats_df = stats_query.collect()
display(stats_df)

: 